<a href="https://colab.research.google.com/github/SaraEstilaei/incident-sentiment-analyzer/blob/Model%2FNLP_04_ensemble/NLP_04_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix

drive.mount("/content/drive")

ROOT  = "/content/drive/MyDrive/NLP_Proj"
D_DIR = f"{ROOT}/deberta_80k"          # خروجی نوت‌بوک DeBERTa
N_DIR = f"{ROOT}/nlptown_80k"          # خروجی کگل (nlptown)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
val_df  = pd.read_csv(f"{D_DIR}/data/val.csv")
test_df = pd.read_csv(f"{D_DIR}/data/test.csv")
y_val, y_test = val_df["label"].values, test_df["label"].values

P = {
    "deberta": {"val": np.load(f"{D_DIR}/val_probs.npy"), "test": np.load(f"{D_DIR}/test_probs.npy")},
    "nlptown": {"val": np.load(f"{N_DIR}/val_probs.npy"), "test": np.load(f"{N_DIR}/test_probs.npy")},
}

for m, d in P.items():
    assert d["val"].shape  == (len(y_val), 5),  f"{m}: shape val اشتباه"
    assert d["test"].shape == (len(y_test), 5), f"{m}: shape test اشتباه"
    assert np.allclose(d["val"].sum(1), 1, atol=1e-3) and np.allclose(d["test"].sum(1), 1, atol=1e-3)

# برچسب‌هایی که nlptown با آن‌ها کار کرده باید دقیقاً همان val.csv و test.csv باشد (یعنی ترتیب ردیف‌ها یکی است)
assert (np.load(f"{N_DIR}/val_labels.npy")  == y_val).all()
assert (np.load(f"{N_DIR}/test_labels.npy") == y_test).all()
print("همه‌ی چک‌ها اوکی: val", len(y_val), "| test", len(y_test))

همه‌ی چک‌ها اوکی: val 16000 | test 8000


In [22]:
def metrics(p, y):
    pred  = p.argmax(1)
    micro = f1_score(y, pred, average="micro")
    return {"micro_F1": micro,
            "macro_F1": f1_score(y, pred, average="macro"),
            "log_loss": log_loss(y, p, labels=range(5)),
            "±95%":     1.96 * np.sqrt(micro * (1 - micro) / len(y))}

def blend(split, w):                     # w = وزن DeBERTa
    return w * P["deberta"][split] + (1 - w) * P["nlptown"][split]

rows = {}
for split, y in [("val", y_val), ("test", y_test)]:
    rows[f"deberta ({split})"]          = metrics(P["deberta"][split], y)
    rows[f"nlptown ({split})"]          = metrics(P["nlptown"][split], y)
    rows[f"ensemble 50/50 ({split})"]   = metrics(blend(split, 0.5), y)
print(pd.DataFrame(rows).T.round(4))

                       micro_F1  macro_F1  log_loss    ±95%
deberta (val)            0.7418    0.5885    0.6445  0.0068
nlptown (val)            0.7243    0.5861    0.7054  0.0069
ensemble 50/50 (val)     0.7456    0.6030    0.6404  0.0067
deberta (test)           0.7369    0.5816    0.6505  0.0096
nlptown (test)           0.7206    0.5810    0.7115  0.0098
ensemble 50/50 (test)    0.7439    0.5978    0.6455  0.0096


In [23]:
ws = np.round(np.arange(0, 1.01, 0.1), 1)
sweep = pd.DataFrame({w: metrics(blend("val", w), y_val) for w in ws}).T[["micro_F1", "macro_F1", "log_loss"]]
sweep.index.name = "وزن DeBERTa"
print(sweep.round(4))

best_w = float(ws[int(sweep["log_loss"].values.argmin())])      # انتخاب وزن فقط با val
print("\nوزن انتخاب‌شده با val (کمترین log-loss):", best_w)
print("test با وزن 0.5     :", {k: round(v, 4) for k, v in metrics(blend("test", 0.5), y_test).items()})
print(f"test با وزن {best_w}:", {k: round(v, 4) for k, v in metrics(blend('test', best_w), y_test).items()})

             micro_F1  macro_F1  log_loss
وزن DeBERTa                              
0.0            0.7243    0.5861    0.7054
0.1            0.7290    0.5888    0.6797
0.2            0.7328    0.5923    0.6644
0.3            0.7371    0.5953    0.6536
0.4            0.7414    0.5984    0.6458
0.5            0.7456    0.6030    0.6404
0.6            0.7458    0.6020    0.6370
0.7            0.7458    0.5983    0.6356
0.8            0.7444    0.5946    0.6361
0.9            0.7432    0.5922    0.6388
1.0            0.7418    0.5885    0.6445

وزن انتخاب‌شده با val (کمترین log-loss): 0.7
test با وزن 0.5     : {'micro_F1': 0.7439, 'macro_F1': 0.5978, 'log_loss': 0.6455, '±95%': np.float64(0.0096)}
test با وزن 0.7: {'micro_F1': 0.742, 'macro_F1': 0.5918, 'log_loss': 0.6408, '±95%': np.float64(0.0096)}


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: T

In [24]:
pd_, pn, pe = (P["deberta"]["test"].argmax(1), P["nlptown"]["test"].argmax(1), blend("test", 0.5).argmax(1))
cd, cn, ce = pd_ == y_test, pn == y_test, pe == y_test

print("هر دو درست        :", round((cd & cn).mean(), 4))
print("فقط DeBERTa درست  :", round((cd & ~cn).mean(), 4))
print("فقط nlptown درست  :", round((~cd & cn).mean(), 4))
print("هر دو غلط         :", round((~cd & ~cn).mean(), 4))

dis = pd_ != pn
print(f"\nتوافق دو مدل: {(~dis).mean():.4f} | دقت وقتی موافق‌اند: {cd[~dis].mean():.4f}")
print(f"وقتی مخالف‌اند ({dis.sum()} نمونه): DeBERTa {cd[dis].mean():.4f} | nlptown {cn[dis].mean():.4f} | ensemble {ce[dis].mean():.4f}")

# آیا بهبود ensemble نسبت به DeBERTa تنها واقعی است؟ (paired bootstrap)
rng = np.random.RandomState(0)
diffs = [ce[i].mean() - cd[i].mean() for i in (rng.randint(0, len(y_test), len(y_test)) for _ in range(2000))]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nبهبود ensemble نسبت به DeBERTa روی test: {ce.mean() - cd.mean():+.4f} | بازه‌ی 95٪: [{lo:+.4f}, {hi:+.4f}]")
print("بهبود معنادار است" if lo > 0 else "بازه شامل صفر است: بهبود قطعی نیست")

هر دو درست        : 0.6647
فقط DeBERTa درست  : 0.0721
فقط nlptown درست  : 0.0559
هر دو غلط         : 0.2072

توافق دو مدل: 0.8383 | دقت وقتی موافق‌اند: 0.7930
وقتی مخالف‌اند (1294 نمونه): DeBERTa 0.4459 | nlptown 0.3454 | ensemble 0.4892

بهبود ensemble نسبت به DeBERTa روی test: +0.0070 | بازه‌ی 95٪: [+0.0012, +0.0130]
بهبود معنادار است


In [25]:
print(classification_report(y_test, pe, target_names=["1", "2", "3", "4", "5"], zero_division=0))
print("Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):")
print(confusion_matrix(y_test, pe))

              precision    recall  f1-score   support

           1       0.67      0.76      0.71       789
           2       0.46      0.29      0.36       542
           3       0.48      0.54      0.51       775
           4       0.63      0.44      0.52      1497
           5       0.85      0.94      0.89      4397

    accuracy                           0.74      8000
   macro avg       0.62      0.59      0.60      8000
weighted avg       0.73      0.74      0.73      8000

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[ 600   83   61   14   31]
 [ 183  158  159   23   19]
 [  80   86  417  129   63]
 [  19    9  188  664  617]
 [  17    4   37  227 4112]]


<div dir="rtl">
حالا میریم سراغ  دیتای Test نهاییمون

</div>

In [26]:
FINAL_DIR = f"{ROOT}/final_test"

final_df = pd.read_csv(f"{FINAL_DIR}/test_cleaned.csv")

P_final = {
    "deberta": np.load(f"{FINAL_DIR}/deberta_final_probs.npy"),
    "nlptown": np.load(f"{FINAL_DIR}/nlptown_final_probs.npy")
}

print("Final data:", final_df.shape)
print("DeBERTa:", P_final["deberta"].shape)
print("NLPTown:", P_final["nlptown"].shape)

Final data: (20000, 13)
DeBERTa: (20000, 5)
NLPTown: (20000, 5)


<div dir="rtl">
بررسی همترازی
</div>

In [27]:
assert len(final_df) == 20000

assert P_final["deberta"].shape == (len(final_df), 5)
assert P_final["nlptown"].shape == (len(final_df), 5)

assert np.allclose(P_final["deberta"].sum(axis=1), 1, atol=1e-3)
assert np.allclose(P_final["nlptown"].sum(axis=1), 1, atol=1e-3)

assert np.isfinite(P_final["deberta"]).all()
assert np.isfinite(P_final["nlptown"]).all()

print("All final prediction checks passed!")

All final prediction checks passed!


In [28]:
final_ensemble_probs = (
    best_w * P_final["deberta"]
    + (1 - best_w) * P_final["nlptown"]
)

print("Ensemble shape:", final_ensemble_probs.shape)
print("First prediction probabilities:", final_ensemble_probs[0])
print("Probability sum:", final_ensemble_probs[0].sum())
print("DeBERTa weight:", best_w)

Ensemble shape: (20000, 5)
First prediction probabilities: [0.85039806 0.09614764 0.03081445 0.01112112 0.0115187 ]
Probability sum: 1.0
DeBERTa weight: 0.7


In [29]:
y_pred = np.argmax(final_ensemble_probs, axis=1) + 1

print("Predictions shape:", y_pred.shape)
print("First 10 predictions:", y_pred[:10])

print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts().sort_index())

Predictions shape: (20000,)
First 10 predictions: [1 1 1 1 1 1 1 1 1 3]

Prediction distribution:
1    5394
2    1886
3    4328
4    2642
5    5750
Name: count, dtype: int64


<div dir="rtl">
مقایسه برای خودمون :)


</div>

In [32]:
for i in range(10):
    deberta_probs = P_final["deberta"][i]
    nlptown_probs = P_final["nlptown"][i]

    deberta_pred = np.argmax(deberta_probs) + 1
    nlptown_pred = np.argmax(nlptown_probs) + 1
    ensemble_pred = np.argmax(final_ensemble_probs[i]) + 1

    print(f"\n{'='*70}")
    print(f"Review {i+1}")
    print("-" * 70)

    print("Text:", final_df.iloc[i]["full_text"][:250], "...")

    print("\nDeBERTa:")
    print("Probabilities:", deberta_probs.round(3))
    print("Prediction:", deberta_pred)

    print("\nNLPTown:")
    print("Probabilities:", nlptown_probs.round(3))
    print("Prediction:", nlptown_pred)

    print("\nEnsemble:")
    print("Probabilities:", final_ensemble_probs[i].round(3))
    print("Prediction:", ensemble_pred)


Review 1
----------------------------------------------------------------------
Text: The prize for most useless invention of all time goes to: FPS01-C ... Conversion Tray for Laptop (Spoiler: its not for a laptop) The name and description of this device are misleading. It is supposed to be a conversion tray for a laptop optical drive ...

DeBERTa:
Probabilities: [0.841 0.106 0.033 0.01  0.009]
Prediction: 1

NLPTown:
Probabilities: [0.872 0.073 0.025 0.013 0.016]
Prediction: 1

Ensemble:
Probabilities: [0.85  0.096 0.031 0.011 0.012]
Prediction: 1

Review 2
----------------------------------------------------------------------
Text: MELTED MOLEX CONNECTOR One of the molex connectors on the power supply started MELTING! It almost burned out my entire motherboard and GPU's. BEWARE! I thought EVGA was a good brand so I picked up one of their PSU's, I won't be making that mistake ag ...

DeBERTa:
Probabilities: [0.857 0.109 0.023 0.007 0.004]
Prediction: 1

NLPTown:
Probabilities: [0.78 

In [30]:
submission = pd.DataFrame({
    "predicted": y_pred
})

submission.to_csv(
    "q2_submission.csv",
    index=False
)

print(submission.head(10))
print("Submission shape:", submission.shape)

   predicted
0          1
1          1
2          1
3          1
4          1
5          1
6          1
7          1
8          1
9          3
Submission shape: (20000, 1)


In [31]:
check = pd.read_csv("q2_submission.csv")

assert check.shape == (20000, 1)
assert check.columns.tolist() == ["predicted"]
assert check["predicted"].between(1, 5).all()
assert check["predicted"].notna().all()

print("All submission checks passed!")
print("File: q2_submission.csv")

All submission checks passed!
File: q2_submission.csv


In [33]:
from google.colab import files

files.download("q2_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>